Problem Statement
We have a sales dataset with this schema:


Task:
 Find the top 3 customers (by total sales amount) in each region over the last 6 months, along with the categories of products they purchased.

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, DateType
from datetime import date, timedelta

sales_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("order_date", DateType(), True)
])

customers_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("region", StringType(), True)
])

products_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True)
])

sales_data = [
    (1, 101, 201, 120.0, date(2026, 5, 1)),
    (2, 102, 202, 250.0, date(2026, 4, 15)),
    (3, 103, 203, 300.0, date(2026, 3, 20)),
    (4, 101, 202, 180.0, date(2026, 2, 10)),
    (5, 104, 204, 90.0, date(2026, 5, 5)),
    (6, 102, 201, 210.0, date(2026, 1, 25)),
    (7, 105, 205, 400.0, date(2026, 4, 30)),
    (8, 103, 204, 150.0, date(2026, 3, 5)),
    (9, 101, 203, 220.0, date(2026, 2, 28)),
    (10, 102, 205, 130.0, date(2026, 5, 8))
]

customers_data = [
    (101, "Alice", "North"),
    (102, "Bob", "South"),
    (103, "Charlie", "East"),
    (104, "Diana", "West"),
    (105, "Eve", "North")
]

products_data = [
    (201, "Laptop", "Electronics"),
    (202, "Tablet", "Electronics"),
    (203, "Desk", "Furniture"),
    (204, "Chair", "Furniture"),
    (205, "Pen", "Stationery")
]

sales = spark.createDataFrame(sales_data, sales_schema)
customers = spark.createDataFrame(customers_data, customers_schema)
products = spark.createDataFrame(products_data, products_schema)

In [0]:
display(sales)
display(customers)
display(products)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
#Filter sales for last six months
last_6_months =  sales.filter(F.col("order_date")>=F.add_months(F.current_date(),-6))
display(last_6_months)

In [0]:
#join with customers and products
joined_df = last_6_months.join(customers,on='customer_id',how='inner').join(products,on="product_id",how="inner")
display(joined_df)

In [0]:
#Aggregate total sales amount by customer + region
agg_df = joined_df.groupBy('region','customer_id','customer_name').agg(F.sum("amount").alias("Total_Sales"),F.collect_set("category").alias("Category_Purschaed"))

In [0]:
display(agg_df)

In [0]:
windowSpec = Window.partitionBy('customer_id').orderBy(F.col('Total_Sales').desc())
#Add rank column
agg_df = agg_df.withColumn('rank',F.row_number().over(windowSpec))
display(agg_df)

In [0]:
#Get Top 3 customers per region
top_customers = agg_df.filter(F.col('rank')<=3).drop("rank")
display(top_customers)
